# Comparison between normal traffic and IBR traffic

This Jupyter notebook compares the characteristics of normal traffic and IBR traffic.

The script uses the conversation output from `tshark` to generate a histogram of the number of packets per conversation. The histogram is then compared between normal traffic.

First:

```sh
tshark -r darknet_20251010.pcap -Y "tcp" -w darknet_20251010_tcp.pcap
tshark -r darknet_20251010.pcap -Y "udp" -w darknet_20251010_udp.pcap

tshark -r darknet_20251010_tcp.pcap -q -z conv,tcp |tee darknet_20251010_tcp_conversations.txt
tshark -r darknet_20251010_udp.pcap -q -z conv,udp |tee darknet_20251010_udp_conversations.txt
```

Then use the script `conversor_tshark_wireshark.py` to convert the conversation output from `tshark` to a pandas DataFrame with the `Wireshark` columns. This is because is possible to genereate the converasation files using the script `plot-conversation-histogram-wireshark.ipynb` too. Using `tshark` is better for big files.

As normal traffic, we are using  `Monday-WorkingHours-stats-tcp.txt` and `Monday-WorkingHours-stats-tcp.txt` files.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# TCP input files
normal_traffic_tcp = 'Monday-WorkingHours-stats-tcp.txt'
normal_traffic_udp = 'Monday-WorkingHours-stats-udp.txt'

# UDP input files
ibr_traffic_tcp = 'darknet_20251010_tcp_conversations.csv'
ibr_traffic_udp = 'darknet_20251010_udp_conversations.csv'

# PDF output files
output_pdf_tcp = 'comparison_flows_generic_ibr_tcp.pdf'
output_pdf_udp = 'comparison_flows_generic_ibr_udp.pdf'

In [ ]:
# These key names are used in the Wireshark conversation statistics
key_1 = 'Packets A → B'
key_2 = 'Packets B → A'

In [ ]:
def plot_packet_histogram(csv_file, ax, title):
    # Read data
    df = pd.read_csv(csv_file, sep=',')

    # Define intervals
    bins = [0, 1, 5, 20, 100, float('inf')]
    labels = ['1', '2-5', '6-20', '21-100', '>100']

    # Classify each value into its interval for both columns
    df['bin_A_B'] = pd.cut(df[key_1], bins=bins, labels=labels,
                           right=True, include_lowest=True)
    df['bin_B_A'] = pd.cut(df[key_2], bins=bins, labels=labels,
                           right=True, include_lowest=True)

    # Calculate totals per bin (summing packets)
    packets_B_A = df.groupby('bin_A_B', observed=False)[key_1].sum()
    packets_A_B = df.groupby('bin_B_A', observed=False)[key_2].sum()

    # Create DataFrame and calculate percentages
    hist_df = (
        pd.DataFrame(
            {
                'Packets received': packets_A_B,
                'Packets replied': packets_B_A
            }
        )
        .fillna(0)
        .reindex(labels)
    )
    total_packets = hist_df['Packets received'].sum() + hist_df['Packets replied'].sum()
    hist_df_percent = hist_df / total_packets * 100

    # Draw stacked chart on the provided axis
    hist_df_percent.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        width=0.85,
        color=['#1f77b4', '#ff7f0e'],
        legend=False  # shared legend outside the function
    )

    # Add labels (inside if total > 60, above if <= 60)
    for idx, (entrada, salida) in enumerate(
        zip(hist_df_percent['Packets received'],
            hist_df_percent['Packets replied'])
    ):
        total = entrada + salida
        label = f"({entrada:.1f}%/{salida:.1f}%)"

        if total > 60:
            # Centrado vertical dentro de la barra total
            y = total / 2.0
            va = 'center'
        else:
            # Justo por encima de la barra
            y = total + 2
            va = 'bottom'

        ax.text(
            idx,
            y,
            label,
            ha='center',
            va=va,
            fontsize=9,
            fontweight='bold',
            rotation=90
        )

    # Styling
    ax.set_ylim(0, 100)  # terminar en 100 %
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Total packets (%)', fontsize=12)
    ax.set_xticklabels(labels, rotation=90, fontsize=12)


In [ ]:
def generate_graphic(file1, title1, file2, title2, output_file):
    # Create figure with two subplots in a row
    fig, axes = plt.subplots(1, 2, figsize=(6, 6), sharey=True)

    # Draw both histograms
    plot_packet_histogram(file1, axes[0], title1)
    plot_packet_histogram(file2, axes[1], title2)

    # Shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, title='')

    # Set the X label for the entire figure
    fig.text(0.5, -0.05, 'Number of aggregated packets', ha='center', fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for legend above

    # Save the figure in the file comparison_flows_generic_ibr_tcp.pdf
    fig.savefig(output_file)

    plt.show()

In [ ]:
def generate_graphic(file1, title1, file2, title2, output_file):
    # Create figure with two subplots in a row
    fig, axes = plt.subplots(1, 2, figsize=(6, 6), sharey=True)

    # Draw both histograms
    plot_packet_histogram(file1, axes[0], title1)
    plot_packet_histogram(file2, axes[1], title2)

    # Shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, title='')

    # Shared X-axis label
    fig.supxlabel('Number of aggregated packets', fontsize=12, y=0.025)

    # Remove duplicated Y-axis label from right axis
    axes[1].set_ylabel('')

    # Compact margins and smaller horizontal spacing
    fig.subplots_adjust(left=0.12, right=0.98, top=0.88, bottom=0.20, wspace=0.06)

    # Save the figure in the file comparison_flows_generic_ibr_tcp.pdf
    fig.savefig(output_file)

    plt.show()

In [ ]:
generate_graphic(normal_traffic_tcp, '(a) Generic traffic', ibr_traffic_tcp, '(b) IBR traffic', output_pdf_tcp)

In [ ]:
generate_graphic(normal_traffic_udp, '(a) Generic traffic', ibr_traffic_udp, '(b) IBR traffic', output_pdf_udp)